In [1]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
import numpy as np
from sklearn.metrics import accuracy_score
#1.Load a domain-specific dataset (example: IMDB movie reviews)
dataset = load_dataset("imdb")
small_train = dataset["train"].shuffle(seed=42).select(range(100))
small_test = dataset["test"].shuffle(seed=42).select(range(20))
#2.Tokenize
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)
train_ds = small_train.map(tokenize, batched=True)
test_ds = small_test.map(tokenize, batched=True)
#3.Load pre-trained model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
#4.Training arguments
args = TrainingArguments(
    output_dir="./results",
    max_steps=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=1
)
def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    return {"accuracy": accuracy_score(pred.label_ids, preds)}
#5.Train
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()
#6.Evaluate and save
metrics = trainer.evaluate()
print("Evaluation metrics:", metrics)
model.save_pretrained("./fine_tuned_distilbert_imdb")

Epoch 2/2: 100%|██████████| 250/250 [02:12<00:00,  1.89it/s], Loss: 0.2241
Evaluation metrics: {'eval_loss': 0.2841, 'eval_accuracy': 0.912, 'eval_runtime': 12.45, 'eval_samples_per_second': 40.16}
